# 01 框架搭建 — 可对话 Agent 框架

把 agent-basics/03 的 Agent Loop 抽成可复用的框架。

**两种模式**：
- **编程模式**：代码中 import Agent，调用 .chat() 执行任务
- **交互模式**：`python cli.py` 启动 REPL 对话

**学习点**：Agent 类设计、消息管理、工具注册、双模式统一

In [ ]:
import sys, json
sys.path.insert(0, '..')

from src.agent_framework import Agent, LLMClient, ConversationMemory, ToolRegistry

print('一切就绪')

## 工具定义

先准备三个演示工具，和 agent-basics 中一样。

In [ ]:
def get_weather(city):
    w = {"北京": {"temp": 25, "desc": "晴朗"}, "上海": {"temp": 28, "desc": "多云"},
         "东京": {"temp": 22, "desc": "小雨"}, "纽约": {"temp": 15, "desc": "阴天"}}
    for k in w:
        if k in city:
            return json.dumps(w[k], ensure_ascii=False)
    return json.dumps({"temp": 20, "desc": "未知"}, ensure_ascii=False)

def search_web(query):
    results = {
        "特斯拉": "特斯拉当前股价 $245，上季度为 $220。",
        "茅台": "茅台当前股价 ¥1650。",
        "图灵奖": "图灵奖是计算机领域最高荣誉，由ACM于1966年设立。",
        "东京人口": "东京都人口约1400万。",
    }
    for k, v in results.items():
        if k in query or query in k:
            return v
    return f"未找到关于'{query}'的结果。"

def calculate(expression):
    try:
        return str(eval(expression))
    except:
        return "计算错误"

DEMO_TOOLS = [
    {"name": "get_weather", "description": "查询指定城市的天气",
     "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]},
     "fn": get_weather},
    {"name": "search_web", "description": "搜索网页获取知识或信息",
     "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
     "fn": search_web},
    {"name": "calculate", "description": "执行数学计算",
     "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]},
     "fn": calculate},
]
print("工具已定义:", [t["name"] for t in DEMO_TOOLS])

## 编程模式 — 创建 Agent 实例

通过构造函数注入 LLM、工具、system prompt。

In [ ]:
agent = Agent(
    tools=DEMO_TOOLS,
    system_prompt="你是一个有用的 AI 助手，可以用中文回复。需要时使用工具获取信息。"
)
print("Agent 已创建，工具数:", len(agent.tools))

### 单工具调用

Agent 会自动判断需要调用哪个工具。

In [ ]:
reply = agent.chat("北京今天天气怎么样？")
print(f"回复: {reply}")

### 多工具并行调用

LLM 可以在同一轮并行调用多个工具。

In [ ]:
reply = agent.chat("北京和上海今天天气分别怎么样？")
print(f"回复: {reply}")

### 多轮工具链（ReAct）

工具的结果会触发新的工具调用。

In [ ]:
reply = agent.chat("特斯拉股价涨了多少？计算涨幅百分比。")
print(f"回复: {reply}")

## 多轮对话（短期记忆）

连续对话中，Agent 记住之前的内容。

In [ ]:
# 新 Agent
agent2 = Agent(tools=DEMO_TOOLS)

r1 = agent2.chat("我叫小明，我在北京。")
print(f"[第1轮] {r1}")
print()

r2 = agent2.chat("我在哪个城市？")
print(f"[第2轮] {r2}")
print()

# 查看记忆状态
stats = agent2.memory.stats()
print(f"消息数: {stats['n_messages']}, 估算 tokens: {stats['estimated_tokens']}")

## Memory 模块独立演示

Memory 可以脱离 Agent 独立使用。

In [ ]:
from src.agent_framework import ConversationMemory

mem = ConversationMemory(system_prompt="你是一个助手")
mem.add_user("你好")
# 模拟 assistant 消息
class FakeMsg:
    content = "你好！有什么可以帮你的？"
    tool_calls = None
mem.add_assistant(FakeMsg())
mem.add_user("天气怎么样？")

print("消息数:", len(mem.get_messages()))
print("统计:", mem.stats())
print()
for m in mem.get_messages():
    print(f"  [{m['role']}] {str(m.get('content', ''))[:80]}")

mem.clear()
print(f"\n清空后消息数: {len(mem.get_messages())}")  # 1 (system prompt 保留)

## 交互模式

运行 `python cli.py` 启动 REPL。下面用编程模式模拟同样的效果：

In [ ]:
# 模拟交互式对话
agent3 = Agent(tools=DEMO_TOOLS, system_prompt="你是一个有用的 AI 助手。")

print("=" * 50)
print("👤 You: 东京天气怎么样？适合出门的话帮我查东京人口")
print("=" * 50)
reply = agent3.chat("东京天气怎么样？适合出门的话帮我查东京人口")
print(f"🤖 Agent: {reply}")

## 总结

| 组件 | 职责 |
|------|------|
| `core.py` | Agent 主循环，组装各模块 |
| `memory.py` | 消息存储、统计、上下文裁剪 |
| `tools.py` | 工具注册、格式输出、安全执行 |
| `llm.py` | LLM API 封装 |
| `cli.py` | 交互式 REPL |

框架核心约 100 行。后续每个实验都在这个框架上**增量添加**，不会推倒重来。